# 🎬 Hybrid Movie Recommendation System

A portfolio project demonstrating a **hybrid recommendation system** that combines:
- **Content-Based Filtering** (TF-IDF + Cosine Similarity)
- **Collaborative Filtering** (KNN, SVD, NMF)
- **Hybrid Approach** (Weighted combination)

**Dataset:** MovieLens Latest Small (100K ratings, 9K+ movies)  
**Author:** [Your Name]  
**Date:** September 2026

---

## 1. Setup & Imports

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Set style
plt.style.use('dark_background')
sns.set_palette('viridis')
pd.set_option('display.max_columns', 20)
pd.set_option('display.max_colwidth', 60)

# Add project root to path
PROJECT_ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, PROJECT_ROOT)

print(f'Project root: {PROJECT_ROOT}')
print(f'Python {sys.version}')

## 2. Download & Load Data

In [ ]:
# Download dataset (idempotent — skips if already present)
from data.download_data import download_dataset
download_dataset()

In [ ]:
from src.data_loader import load_movies, load_ratings, load_tags, get_dataset_stats, get_sparsity, create_user_item_matrix

movies = load_movies()
ratings = load_ratings()
tags = load_tags()

print(f'Movies:  {movies.shape}')
print(f'Ratings: {ratings.shape}')
print(f'Tags:    {tags.shape}')

In [ ]:
movies.head()

In [ ]:
ratings.head()

In [ ]:
stats = get_dataset_stats(movies, ratings, tags)
for key, value in stats.items():
    print(f'{key:>25}: {value}')

## 3. Exploratory Data Analysis (EDA)

### 3.1 Rating Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Rating value distribution
rating_counts = ratings['rating'].value_counts().sort_index()
axes[0].bar(rating_counts.index.astype(str), rating_counts.values, 
            color='#667eea', edgecolor='white', linewidth=0.5)
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Ratings', fontweight='bold')

# Rating histogram (continuous)
axes[1].hist(ratings['rating'], bins=10, color='#764ba2', 
             edgecolor='white', linewidth=0.5, alpha=0.8)
axes[1].axvline(ratings['rating'].mean(), color='#f093fb', 
                linestyle='--', linewidth=2, label=f'Mean: {ratings["rating"].mean():.2f}')
axes[1].set_xlabel('Rating')
axes[1].set_ylabel('Count')
axes[1].set_title('Rating Histogram', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Mean rating: {ratings["rating"].mean():.3f}')
print(f'Median rating: {ratings["rating"].median()}')
print(f'Std dev: {ratings["rating"].std():.3f}')

### 3.2 Genre Analysis

In [ ]:
# Genre frequency
all_genres = [genre for gl in movies['genre_list'] for genre in gl]
genre_counts = pd.Series(Counter(all_genres)).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(genre_counts)))
ax.barh(genre_counts.index, genre_counts.values, color=colors, edgecolor='white', linewidth=0.5)
ax.set_xlabel('Number of Movies')
ax.set_title('Movies per Genre', fontweight='bold', fontsize=14)

# Add count labels
for i, (val, name) in enumerate(zip(genre_counts.values, genre_counts.index)):
    ax.text(val + 20, i, str(val), va='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Average rating per genre
genre_ratings = []
for _, row in movies.iterrows():
    movie_ratings = ratings[ratings['movieId'] == row['movieId']]['rating']
    if len(movie_ratings) > 0:
        avg = movie_ratings.mean()
        for genre in row['genre_list']:
            genre_ratings.append({'genre': genre, 'avg_rating': avg})

genre_rating_df = pd.DataFrame(genre_ratings)
genre_avg = genre_rating_df.groupby('genre')['avg_rating'].mean().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
colors = plt.cm.plasma(np.linspace(0.2, 0.9, len(genre_avg)))
ax.barh(genre_avg.index, genre_avg.values, color=colors, edgecolor='white', linewidth=0.5)
ax.set_xlabel('Average Rating')
ax.set_title('Average Rating per Genre', fontweight='bold', fontsize=14)
ax.set_xlim(2.5, 4.5)
plt.tight_layout()
plt.show()

### 3.3 User & Movie Activity

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Ratings per user
user_activity = ratings.groupby('userId').size()
axes[0].hist(user_activity, bins=50, color='#667eea', edgecolor='white', linewidth=0.5)
axes[0].set_xlabel('Number of Ratings')
axes[0].set_ylabel('Number of Users')
axes[0].set_title('User Activity Distribution', fontweight='bold')
axes[0].axvline(user_activity.mean(), color='#f093fb', linestyle='--', 
                label=f'Mean: {user_activity.mean():.0f}')
axes[0].legend()

# Ratings per movie
movie_popularity = ratings.groupby('movieId').size()
axes[1].hist(movie_popularity, bins=50, color='#764ba2', edgecolor='white', linewidth=0.5)
axes[1].set_xlabel('Number of Ratings')
axes[1].set_ylabel('Number of Movies')
axes[1].set_title('Movie Popularity Distribution', fontweight='bold')
axes[1].axvline(movie_popularity.mean(), color='#f093fb', linestyle='--',
                label=f'Mean: {movie_popularity.mean():.0f}')
axes[1].legend()

plt.tight_layout()
plt.show()

### 3.4 Sparsity Analysis

In [ ]:
user_item_matrix = create_user_item_matrix(ratings)
sparsity = get_sparsity(user_item_matrix)

print(f'User-Item Matrix Shape: {user_item_matrix.shape}')
print(f'Total possible ratings: {user_item_matrix.shape[0] * user_item_matrix.shape[1]:,}')
print(f'Actual ratings: {user_item_matrix.notna().sum().sum():,}')
print(f'Sparsity: {sparsity:.2f}%')
print()
print('→ The matrix is very sparse, which is typical for recommendation systems.')
print('  This is why collaborative filtering uses dimensionality reduction (SVD/NMF).')

---

## 4. Content-Based Filtering

**Approach:** Represent each movie as a TF-IDF vector of its genres and tags, then use cosine similarity to find the most similar movies.

**Why TF-IDF?** It weights features by how distinctive they are — a common genre like "Drama" gets a lower weight than a niche genre like "Film-Noir".

In [ ]:
from src.content_based import ContentBasedRecommender

content_rec = ContentBasedRecommender()
content_rec.fit(movies, tags)

print(f'TF-IDF Matrix Shape: {content_rec.tfidf_matrix.shape}')
print(f'Vocabulary Size: {len(content_rec.tfidf.get_feature_names_out())}')
print(f'Similarity Matrix Shape: {content_rec.similarity_matrix.shape}')

### 4.1 Item-to-Item Recommendations

Find movies similar to a given movie based on content features.

In [ ]:
# Find movies similar to Toy Story (movieId=1)
print('Movies similar to "Toy Story (1995)":\n')
similar = content_rec.recommend_similar(movie_id=1, n=10)
similar

In [ ]:
# Try another movie: The Matrix (movieId=2571)
print('Movies similar to "The Matrix (1999)":\n')
similar_matrix = content_rec.recommend_similar(movie_id=2571, n=10)
similar_matrix

In [ ]:
# Inspect TF-IDF features for The Matrix
features = content_rec.get_movie_features(2571)
print('Top features for "The Matrix":')
for feat, weight in list(features.items())[:10]:
    print(f'  {feat:20s} → {weight:.4f}')

### 4.2 User-Level Recommendations

Aggregate content similarities from movies a user has liked to generate personalized recommendations.

In [ ]:
# Recommendations for User 1
user_id = 1
user_rated = ratings[ratings['userId'] == user_id].merge(movies[['movieId', 'title', 'genres']], on='movieId')
print(f'User {user_id} has rated {len(user_rated)} movies.')
print(f'Top rated movies:')
user_rated.sort_values('rating', ascending=False).head(5)[['title', 'genres', 'rating']]

In [ ]:
content_recs = content_rec.recommend_for_user(user_id, ratings, n=10)
print(f'\nContent-Based Recommendations for User {user_id}:\n')
content_recs

---

## 5. Collaborative Filtering

Uses patterns in user-item interactions to make predictions. We compare three algorithms:

| Algorithm | Type | Description |
|-----------|------|-------------|
| **KNN Basic** | Memory-based | Finds similar users, predicts from their ratings |
| **SVD** | Model-based | Factorizes the rating matrix into latent factors |
| **NMF** | Model-based | Like SVD, but with non-negativity constraints |

In [ ]:
from src.collaborative import CollaborativeRecommender, compare_algorithms

### 5.1 Cross-Validation Comparison

Compare all three algorithms using 5-fold cross-validation.

In [ ]:
print('Running 5-fold cross-validation for all algorithms...')
print('This may take 30-60 seconds.\n')

comparison = compare_algorithms(ratings, cv=5)
comparison

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#667eea', '#764ba2', '#f093fb']

# RMSE
axes[0].bar(comparison['algorithm'], comparison['rmse_mean'], 
            yerr=comparison['rmse_std'], color=colors, capsize=5,
            edgecolor='white', linewidth=0.5)
axes[0].set_title('RMSE (lower is better)', fontweight='bold')
axes[0].set_ylabel('RMSE')

# MAE
axes[1].bar(comparison['algorithm'], comparison['mae_mean'], 
            yerr=comparison['mae_std'], color=colors, capsize=5,
            edgecolor='white', linewidth=0.5)
axes[1].set_title('MAE (lower is better)', fontweight='bold')
axes[1].set_ylabel('MAE')

plt.tight_layout()
plt.show()

best = comparison.loc[comparison['rmse_mean'].idxmin()]
print(f'\n✓ Best algorithm: {best["algorithm"]} (RMSE: {best["rmse_mean"]:.4f})')

### 5.2 SVD Recommendations

Use the best-performing algorithm (typically SVD) for recommendations.

In [ ]:
svd_rec = CollaborativeRecommender('svd')
svd_rec.fit(ratings)

# Recommendations for User 1
collab_recs = svd_rec.recommend_for_user(user_id=1, ratings_df=ratings, movies_df=movies, n=10)
print('SVD Recommendations for User 1:\n')
collab_recs

---

## 6. Hybrid Recommendation

Combine content-based and collaborative filtering using a weighted hybrid:

$$\text{hybrid\_score} = \alpha \times \text{content\_score} + (1 - \alpha) \times \text{collab\_score}$$

Where $\alpha = 0.3$ by default (collaborative filtering dominates, but content-based fills cold-start gaps).

In [ ]:
from src.hybrid import HybridRecommender

hybrid_rec = HybridRecommender(alpha=0.3, collab_algo='svd')
hybrid_rec.fit(ratings, movies, tags)

In [ ]:
# Hybrid recommendations for User 1
hybrid_recs = hybrid_rec.recommend(user_id=1, n=10)
print('Hybrid Recommendations for User 1:\n')
hybrid_recs

In [ ]:
# Explain the top recommendation
if len(hybrid_recs) > 0:
    top_movie_id = int(hybrid_recs.iloc[0]['movieId'])
    explanation = hybrid_rec.explain(user_id=1, movie_id=top_movie_id)
    
    print(f'Explanation for "{explanation["movie"]}":')
    print(f'  Genres: {explanation["genres"]}')
    print(f'  Predicted Rating: {explanation["predicted_rating"]}/5')
    print(f'  Similar to {len(explanation["similar_to_liked_movies"])} of your liked movies:')
    for m in explanation['similar_to_liked_movies'][:3]:
        print(f'    - {m["title"]} (similarity: {m["similarity"]:.3f})')
    print(f'\n  → {explanation["explanation"]}')

### 6.1 Effect of Alpha (α) on Recommendations

In [ ]:
# Compare recommendations at different alpha values
alphas = [0.0, 0.3, 0.5, 0.7, 1.0]
print('Top 5 recommendations at different α values for User 1:\n')

for a in alphas:
    recs = hybrid_rec.recommend(user_id=1, n=5, alpha=a)
    titles = recs['title'].tolist() if len(recs) > 0 else []
    label = 'Pure Collab' if a == 0 else ('Pure Content' if a == 1 else f'α={a}')
    print(f'  {label:15s} → {", ".join(titles[:3])}...')

---

## 7. Full Evaluation

Comprehensive evaluation with Precision@K, Recall@K, Coverage, RMSE, and MAE.

In [ ]:
from src.evaluation import full_comparison

print('Running full evaluation (Precision@10, Recall@10, Coverage)...')
print('This may take 1-2 minutes.\n')

eval_results = full_comparison(ratings, k=10, n_folds=5)
eval_results

In [ ]:
# Comprehensive visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colors = ['#667eea', '#764ba2', '#f093fb']

metrics = [
    ('rmse', 'RMSE (lower is better)'),
    ('mae', 'MAE (lower is better)'),
    ('precision@10', 'Precision@10 (higher is better)'),
    ('recall@10', 'Recall@10 (higher is better)'),
]

for ax, (metric, title) in zip(axes.flatten(), metrics):
    if metric in eval_results.columns:
        bars = ax.bar(eval_results['algorithm'], eval_results[metric],
                     color=colors, edgecolor='white', linewidth=0.5)
        ax.set_title(title, fontweight='bold')
        ax.set_ylabel(metric.upper())
        
        # Add value labels
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.4f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

---

## 8. Conclusions

### Key Findings

1. **SVD** typically achieves the lowest RMSE/MAE, confirming its effectiveness for matrix factorization on sparse data
2. **Content-based filtering** is effective for item-similarity queries and cold-start scenarios
3. The **hybrid approach** combines the strengths of both methods, particularly for users with sparse rating histories
4. The MovieLens dataset has ~98% sparsity, which makes collaborative filtering challenging but SVD handles it well through dimensionality reduction

### Strengths
- Hybrid approach handles cold-start gracefully
- Explainable recommendations (why was this recommended?)
- Multiple algorithms compared with proper cross-validation

### Potential Improvements
- **Deep Learning**: Neural Collaborative Filtering (NCF) could capture more complex user-item interactions
- **Implicit Feedback**: Use watch time, clicks, etc. instead of just ratings
- **Contextual Features**: Time of day, user demographics, seasonal trends
- **A/B Testing**: Evaluate recommendations with real users
- **Scalability**: Use approximate nearest neighbors (Annoy, FAISS) for large-scale deployment